# Day 8: Snowflake Storage Integration & Dynamic Tables Mock

This notebook uses DuckDB to simulate Snowflake's query behavior and dynamic table transformations over Iceberg-like structured data.

In [ ]:
!pip install duckdb -q

In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect(database=':memory:')
print("DuckDB connected.")

In [ ]:
# 1. Setup Mock Silver Table
conn.execute("""
    CREATE TABLE silver_omop_condition (
        condition_occurrence_id VARCHAR,
        person_id VARCHAR,
        condition_concept_id INT,
        condition_start_date DATE,
        condition_end_date DATE
    );
""")

conn.execute("""
    INSERT INTO silver_omop_condition VALUES 
    ('C1', 'P123', 31967, '2023-10-01', NULL),
    ('C2', 'P456', 31967, '2023-10-01', '2023-10-02'),
    ('C3', 'P789', 32020, '2023-10-02', NULL);
""")

print("Mock Silver OMOP Condition Table:")
print(conn.execute("SELECT * FROM silver_omop_condition").df())

In [ ]:
# 2. Simulate Silver Dynamic Table (Active Conditions)
conn.execute("""
    CREATE VIEW dt_active_conditions AS
    SELECT 
      condition_occurrence_id,
      person_id,
      condition_concept_id,
      condition_start_date
    FROM silver_omop_condition
    WHERE condition_end_date IS NULL;
""")

print("\nSimulated Silver Dynamic Table (dt_active_conditions):")
print(conn.execute("SELECT * FROM dt_active_conditions").df())

In [ ]:
# 3. Simulate Gold Dynamic Table (Aggregated Stats)
conn.execute("""
    CREATE VIEW dt_daily_condition_stats AS
    SELECT
      condition_concept_id,
      condition_start_date AS condition_date,
      COUNT(DISTINCT person_id) AS unique_patients
    FROM dt_active_conditions
    GROUP BY 1, 2;
""")

print("\nSimulated Gold Dynamic Table (dt_daily_condition_stats):")
print(conn.execute("SELECT * FROM dt_daily_condition_stats").df())